In [1]:
import pandas as pd
import psutil

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

print("Free RAM (GB):", round(psutil.virtual_memory().available / 1e9, 1))

prev = pd.read_csv("../data/previous_application.csv")
cust = pd.read_csv("../data/application_train.csv")
desc = pd.read_csv("../data/HomeCredit_columns_description.csv", encoding="latin-1")

# 1. Shapes
print("\n=== SHAPES ===")
print("previous_application:", prev.shape)
print("application_train   :", cust.shape)
print("columns_description :", desc.shape)
print("prev memory (MB):", round(prev.memory_usage(deep=True).sum() / 1e6))

# 2. Columns and types
print("\n=== previous_application: dtypes + % missing ===")
info = pd.DataFrame({"dtype": prev.dtypes,
                     "pct_missing": (prev.isna().mean() * 100).round(1),
                     "n_unique": prev.nunique()})
print(info)

print("\n=== application_train: first 20 columns ===")
print(list(cust.columns[:20]), "... total", cust.shape[1])

# 3. Target column
print("\n=== NAME_CONTRACT_STATUS counts ===")
print(prev["NAME_CONTRACT_STATUS"].value_counts())
print((prev["NAME_CONTRACT_STATUS"].value_counts(normalize=True) * 100).round(2))

# 4. Duplicates and keys
print("\n=== DUPLICATES / KEYS ===")
print("duplicate rows in prev:", prev.duplicated().sum())
print("SK_ID_PREV unique?:", prev["SK_ID_PREV"].is_unique)
print("SK_ID_CURR unique in cust?:", cust["SK_ID_CURR"].is_unique)

# 5. Join coverage
print("\n=== JOIN COVERAGE ===")
in_cust = prev["SK_ID_CURR"].isin(cust["SK_ID_CURR"])
print("prev rows whose customer is in application_train:", in_cust.sum(), "of", len(prev),
      f"({in_cust.mean()*100:.1f}%)")

# 6. Overlapping column names (these cause _x/_y problems in merge)
overlap = sorted(set(prev.columns) & set(cust.columns) - {"SK_ID_CURR"})
print("\n=== COLUMNS IN BOTH TABLES (rename before merging) ===")
print(overlap)

# 7. Time and odd values
print("\n=== DAYS_DECISION ===")
print(prev["DAYS_DECISION"].describe())
print("rows with DAYS_DECISION > -365 (OOT):", (prev["DAYS_DECISION"] > -365).sum())
days_cols = [c for c in prev.columns if c.startswith("DAYS_")]
print("\n=== count of 365243 in prev DAYS_* columns ===")
for c in days_cols:
    print(c, (prev[c] == 365243).sum())
print("\n=== count of XNA / XAP in prev text columns ===")
for c in prev.select_dtypes("object").columns:
    n = prev[c].isin(["XNA", "XAP"]).sum()
    if n > 0:
        print(c, n)

# 8. Data dictionary for previous_application
print("\n=== DATA DICTIONARY (previous_application) ===")
print(desc.columns.tolist())
tbl_col = desc.columns[desc.columns.str.contains("Table", case=False)][0]
d = desc[desc[tbl_col].astype(str).str.contains("previous", case=False)]
print(d.iloc[:, 2:5].to_string())

Free RAM (GB): 2.2

=== SHAPES ===
previous_application: (1670214, 37)
application_train   : (307511, 121)
columns_description : (219, 5)
prev memory (MB): 704

=== previous_application: dtypes + % missing ===
                               dtype  pct_missing  n_unique
SK_ID_PREV                     int64          0.0   1670214
SK_ID_CURR                     int64          0.0    338857
NAME_CONTRACT_TYPE               str          0.0         4
AMT_ANNUITY                  float64         22.3    357959
AMT_APPLICATION              float64          0.0     93885
AMT_CREDIT                   float64          0.0     86803
AMT_DOWN_PAYMENT             float64         53.6     29278
AMT_GOODS_PRICE              float64         23.1     93885
WEEKDAY_APPR_PROCESS_START       str          0.0         7
HOUR_APPR_PROCESS_START        int64          0.0        24
FLAG_LAST_APPL_PER_CONTRACT      str          0.0         2
NFLAG_LAST_APPL_IN_DAY         int64          0.0         2
RATE_DOWN_

C:\Users\anand\AppData\Local\Temp\ipykernel_10496\1443577234.py:62: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in prev.select_dtypes("object").columns:


NAME_PAYMENT_TYPE 627384
CODE_REJECT_REASON 1358337
NAME_CLIENT_TYPE 1941
NAME_GOODS_CATEGORY 950809
NAME_PORTFOLIO 372230
NAME_PRODUCT_TYPE 1063666
NAME_SELLER_INDUSTRY 855720
NAME_YIELD_GROUP 517215

=== DATA DICTIONARY (previous_application) ===
['Unnamed: 0', 'Table', 'Row', 'Description', 'Special']
                             Row                                                                                                                                                                                                                                                          Description                                Special
173                  SK_ID_PREV                                ID of previous credit in Home credit related to loan in our sample. (One loan in our sample can have 0,1,2 or more previous loan applications in Home Credit, previous application could, but not necessarily have to lead to credit)                                  hashed
174                   SK_ID_